In [1]:
# DASHBOARD NOTEBOOK

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json
import runpy
import re
import gc

# GENERAL PARAMETERS
RAW_ROOT =  Path("data/raw_signal")
RAW_EMG_DIR   = RAW_ROOT / "emg"
RAW_BIA_DIR   = RAW_ROOT / "bia"
RAW_NIRS_DIR  = RAW_ROOT / "nirs"
RAW_MYOTON_DIR = RAW_ROOT / "myoton"

## LABELING PARAMETERS
# PROTOCOLE SEQUENCE ORDER 
SEQ_ORDER = "INIT WU REST0 MVC_REF REST1 SVC_REF REST2 EX_DYN REST3 MVC_RECOV_DYN REST4 EX_STA REST5 MVC_RECOV_STA END".split()




## SELECT THE RUN_ID

In [2]:

RUN_ID = "011BeSa_20251023" # EDIT ONLY THIS LINE TO CHANGE THE RUN ID

########

# SET UP CACHE PATHS PARAMETERS
BASE_DERIVED = Path("data/derived") / RUN_ID
CACHE_DIR = BASE_DERIVED / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True) #create cache directory if it doesn't exist

# INDIVIDUAL CACHE PATH
CACHE_01_PARTICIPANTS = CACHE_DIR / "01_participants.parquet"

CACHE_02_EMG_IMPORT = CACHE_DIR / "02_emg_import.parquet"
CACHE_02_MASTER = CACHE_DIR / "02_master.parquet"
CACHE_02_EMG = CACHE_DIR / "02_emg.parquet"
CACHE_02_TORQUE = CACHE_DIR / "02_torque_import.parquet"
CACHE_02_METADATA = CACHE_DIR / "02_metadata.parquet"

CACHE_03a_SEQUENCE_PICKER = CACHE_DIR / "03a_sequence_picked.json"

CACHE_04_VC_KNOBS_EVENTS = CACHE_DIR / "04_vc_knobs_events.parquet"


CACHE_05_BIA_IMPORT = CACHE_DIR / "05_bia_import.parquet"

CACHE_06_BIA_SYNC = CACHE_DIR / "06_bia_sync.parquet"



CACHE_04_NIRS_IMPORT = CACHE_DIR / "04_nirs_import.parquet"
CACHE_05_MYOTON_IMPORT = CACHE_DIR / "05_myoton_import.parquet"

## 1. PARTICIPANT

In [3]:
# ACTIVE SUBJECT (single run per participant)
# ============================================================
#CHECK FOR CACHE PRESENT
if CACHE_01_PARTICIPANTS.exists():
    participants_df = pd.read_parquet(CACHE_01_PARTICIPANTS)
else:
    %run sources/01_participants.py #run the source script to create participants_df
    participants_df.to_parquet(CACHE_01_PARTICIPANTS, index=False) # save to cache for future runs
##



participant_id = RUN_ID.split("_", 1)[0]

participant_match = (participants_df["participant_id"] == participant_id)
assert participant_match.sum() == 1, (
    f"Expected exactly 1 match for participant_id={participant_id}, "
    f"got {participant_match.sum()}"
)

participant_row = participants_df.loc[participant_match].iloc[0]



## 2. EMG

In [4]:
# 02 EMG IMPORT 

if CACHE_02_MASTER.exists():
    master_index_grid = pd.read_parquet(CACHE_02_MASTER)
    emg_compact_df = pd.read_parquet(CACHE_02_EMG)
    torque_compact_df = pd.read_parquet(CACHE_02_TORQUE)

    with open(CACHE_02_METADATA, "r", encoding="utf-8") as f:
        meta = json.load(f)
    ts_ref = pd.to_datetime(meta["ts_ref"])


else:
    local_namespace = runpy.run_path(
        "sources/02_emg_import.py",
        init_globals={
            "RUN_ID": RUN_ID,
            "RAW_ROOT": RAW_ROOT,
            "RAW_EMG_DIR": RAW_EMG_DIR,
            "participant_row": participant_row,
        },
    )

    master_index_grid = local_namespace["master_index_grid"]
    emg_compact_df = local_namespace["emg_compact_df"]
    torque_compact_df = local_namespace["torque_compact_df"]
    ts_ref = local_namespace["ts_ref"]

    del local_namespace
    gc.collect()

    master_index_grid.to_parquet(CACHE_02_MASTER, index=False)
    emg_compact_df.to_parquet(CACHE_02_EMG, index=False)
    torque_compact_df.to_parquet(CACHE_02_TORQUE, index=False)

    with open(CACHE_02_METADATA, "w", encoding="utf-8") as f:
        json.dump({"ts_ref": str(ts_ref)}, f)



## 03. Labeling

In [5]:
# 03.a : MANUAL SEQUENCING THE RESEARCH PROTOCOLE USING EMG DATA (TORQUE)
assert SEQ_ORDER[0] == "INIT" and SEQ_ORDER[-1] == "END"
%matplotlib widget

assert "CACHE_DIR" in globals()
CACHE_03A_SEQUENCE_PICKER = CACHE_DIR / "03a_sequence_picked.json"

if not CACHE_03A_SEQUENCE_PICKER.exists():
    runpy.run_path(
        "sources/03a_sequence_picker.py",
        init_globals={
            "RUN_ID": RUN_ID,
            "CACHE_DIR": CACHE_DIR,
            "master_index_grid": master_index_grid,
            "torque_compact_df": torque_compact_df,
            "SEQ_ORDER": SEQ_ORDER,
            "ts_ref": ts_ref,
        },
    )
else:
    print("[SKIP] 03a_sequence_picked.json exists:", RUN_ID, " / cache /", CACHE_03A_SEQUENCE_PICKER.name)


[SKIP] 03a_sequence_picked.json exists: 011BeSa_20251023  / cache / 03a_sequence_picked.json


In [6]:
# 03b. Labeling
%matplotlib inline 
# revert back to inline for following steps

if "SEQ" in master_index_grid.columns:
    print("[SKIP] SEQ already present in master_index_grid.")
else:
    local_namespace = runpy.run_path(
        "sources/03b_sequence_labeling.py",
        init_globals={
            "RUN_ID": RUN_ID,
            "CACHE_DIR": CACHE_DIR,
            "master_index_grid": master_index_grid,
            "SEQ_ORDER": SEQ_ORDER,
        },
    )

    master_index_grid = local_namespace["master_index_grid"]

    del local_namespace
    gc.collect()

    print("[DONE] 03b_seq_labeling applied.")


SEQ labels present: ['INIT', 'WU', 'REST0', 'MVC_REF', 'REST1', 'SVC_REF', 'REST2', 'EX_DYN', 'REST3', 'MVC_RECOV_DYN', 'REST4', 'EX_STA', 'REST5', 'MVC_RECOV_STA', 'END']
Applied boundaries: 14 Expected: 14
[DONE] 03b_seq_labeling applied.


## 04. VOLUNTARY CONTRACTION DETECTION (VC)

In [7]:
# 04a VC DETECTION — TUNE (knobs in notebook)

import runpy, gc
import pandas as pd

CACHE_04_VC_KNOBS_EVENTS = CACHE_DIR / "04_vc_knobs_events.parquet"

# Skip if cache already has events
skip = False
if CACHE_04_VC_KNOBS_EVENTS.exists():
    df = pd.read_parquet(CACHE_04_VC_KNOBS_EVENTS)
    if ("row_type" in df.columns) and df["row_type"].eq("event").any():
        print("[SKIP] 04 VC: cache already contains events.")
        skip = True

# knobs - EDIT THOSE AND RERUN CELL UNTIL VC ARE PERFECTLY DETECTED 
VC_KNOBS_BY_SEQ = {
        "WU":            {"thr_frac": 0.25, "merge_gap_s": 0.85, "min_dur_s": 3.00},
        "MVC_REF":       {"thr_frac": 0.40, "merge_gap_s": 0.10, "min_dur_s": 3.00},
        "SVC_REF":       {"thr_frac": 0.25, "merge_gap_s": 0.25, "min_dur_s": 3.00},
        "EX_DYN":        {"thr_frac": 0.25, "merge_gap_s": 0.25, "min_dur_s": 1.50},
        "MVC_RECOV_DYN": {"thr_frac": 0.40, "merge_gap_s": 0.10, "min_dur_s": 3.00},
        "EX_STA":        {"thr_frac": 0.25, "merge_gap_s": 0.65, "min_dur_s": 50.00},
        "MVC_RECOV_STA": {"thr_frac": 0.40, "merge_gap_s": 0.10, "min_dur_s": 3.00},
}

if not skip:
    local_namespace = runpy.run_path(
        "sources/04_vc_detection.py",
        init_globals={
            "RUN_ID": RUN_ID,
            "master_index_grid": master_index_grid,
            "torque_compact_df": torque_compact_df,
            "CACHE_04_VC_KNOBS_EVENTS": CACHE_04_VC_KNOBS_EVENTS,
            "VC_KNOBS_BY_SEQ": VC_KNOBS_BY_SEQ,  # <- overrides for this run
        },
    )

    master_index_grid = local_namespace["master_index_grid_out"]
    vc_events_df = local_namespace["vc_events_df"]

    del local_namespace
    gc.collect()


[SKIP] 04 VC: cache already contains events.


In [8]:
# 04b VC DETECTION — COMMIT (writes cache)

import runpy, gc

CACHE_04_VC_KNOBS_EVENTS = CACHE_DIR / "04_vc_knobs_events.parquet"

# Use the same VC_KNOBS_BY_SEQ currently in your notebook (optional)
VC_KNOBS_BY_SEQ = globals().get("VC_KNOBS_BY_SEQ", None)

local_namespace = runpy.run_path(
    "sources/04_vc_detection.py",
    init_globals={
        "RUN_ID": RUN_ID,
        "master_index_grid": master_index_grid,
        "torque_compact_df": torque_compact_df,
        "CACHE_04_VC_KNOBS_EVENTS": CACHE_04_VC_KNOBS_EVENTS,
        "VC_KNOBS_BY_SEQ": VC_KNOBS_BY_SEQ,
        "VC_COMMIT": True,
        "VC_PLOT": False,  # skip plotting during commit for speed
    },
)

master_index_grid = local_namespace["master_index_grid_out"]
vc_events_df = local_namespace["vc_events_df"]

del local_namespace
gc.collect()

print("[OK] 04 VC committed.")


[OK] 04 VC committed.


## 5. BIA

In [9]:
# 5a. BIA IMPORT

local_namespace = runpy.run_path(
    "sources/05_bia_import.py",
    init_globals={
        "RUN_ID": RUN_ID,
        "ts_ref": ts_ref,
        "RAW_BIA_DIR": RAW_BIA_DIR,
        "CACHE_05_BIA_IMPORT": CACHE_05_BIA_IMPORT,
        "master_index_grid": master_index_grid,
    },
)

bia2_compact_df = local_namespace["bia2_compact_df_out"]
bia4_compact_df = local_namespace["bia4_compact_df_out"]
bia2_freqs_hz = local_namespace["bia2_freqs_hz_out"]
bia4_freqs_hz = local_namespace["bia4_freqs_hz_out"]

# Optional: keep the dict if you like, otherwise drop it
bia_tables = local_namespace["bia_tables_out"]

# Cleanup: remove transient variables (no globals sweep)
del local_namespace
gc.collect()


[05_bia_import] RUN_ID=011BeSa_20251023
[05_bia_import] ts_ref = 2025-10-23 14:40:12.250000
[05_bia_import] RAW_BIA_DIR = data\raw_signal\bia
[05_bia_import] Loading 2PT = 011BeSa_20251023_BIA_expe_Z2PT.pkl
[05_bia_import] Loading 4PT = 011BeSa_20251023_BIA_expe_Z4PT.pkl
[05_bia_import] cache hit -> loading parquet siblings


98

In [10]:
# 5b. BIA SYNC

# knobs - EDIT THOSE AND RERUN CELL UNTIL SYNC LOOKS GOOD 
BIA_SYNC_ANCHOR_SEQ = "MVC_REF"
BIA_SYNC_TARGET_HZ  = 9760.0
BIA_SYNC_LAG_MIN_S  = -5.0
BIA_SYNC_LAG_MAX_S  =  5.0
BIA_SYNC_LAG_STEP_S =  0.02
BIA_SYNC_MANUAL_NUDGE_S = 0.0
BIA_SYNC_PLOT   = True
BIA_SYNC_COMMIT = False

CACHE_06_BIA_SYNC = CACHE_DIR / "06_bia_sync.parquet"

local_namespace = runpy.run_path(
    "sources/06_bia_sync.py",
    init_globals={
        "RUN_ID": RUN_ID,
        "master_index_grid": master_index_grid,

        "bia2_compact_df": bia2_compact_df,
        "bia4_compact_df": bia4_compact_df,
        "bia4_freqs_hz": bia4_freqs_hz,

        "torque_compact_df": torque_compact_df,
        "TORQUE_COL": "torque_raw",

        "CACHE_06_BIA_SYNC": CACHE_06_BIA_SYNC,

        "BIA_SYNC_ANCHOR_SEQ": BIA_SYNC_ANCHOR_SEQ,
        "BIA_SYNC_TARGET_HZ": BIA_SYNC_TARGET_HZ,
        "BIA_SYNC_LAG_MIN_S": BIA_SYNC_LAG_MIN_S,
        "BIA_SYNC_LAG_MAX_S": BIA_SYNC_LAG_MAX_S,
        "BIA_SYNC_LAG_STEP_S": BIA_SYNC_LAG_STEP_S,
        "BIA_SYNC_MANUAL_NUDGE_S": BIA_SYNC_MANUAL_NUDGE_S,

        "BIA_SYNC_PLOT": BIA_SYNC_PLOT,
        "BIA_SYNC_COMMIT": BIA_SYNC_COMMIT,
    }
)

bia2_compact_aligned_df = local_namespace["bia2_compact_aligned_df_out"]
bia4_compact_aligned_df = local_namespace["bia4_compact_aligned_df_out"]
bia_sync_report_df      = local_namespace["bia_sync_report_df_out"]


In [11]:
# 5c. BIA SYNC — COMMIT (no recompute)

cache_dir = CACHE_06_BIA_SYNC.parent
cache_dir.mkdir(parents=True, exist_ok=True)

bia2_compact_aligned_df.to_parquet(cache_dir / "06_bia2_compact_aligned.parquet", index=False)
bia4_compact_aligned_df.to_parquet(cache_dir / "06_bia4_compact_aligned.parquet", index=False)
bia_sync_report_df.to_parquet(cache_dir / "06_bia_sync_report.parquet", index=False)

print("Step 06 committed.")



Step 06 committed.


## 4. NIRS

## 5. MYOTON

## 6. QC PLOTS

## 7. DATA EXPORT